In [4]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Dane
data = pd.DataFrame({
    'Transakcja': ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10'],
    'Cena': [45, 150, 20, 75, 300, 30, 120, 25, 50, 200],
    'Liczba produktów': [3, 1, 5, 2, 1, 4, 1, 6, 3, 2],
    'Kategoria': ['Elektronika', 'Meble', 'Żywność', 'Elektronika', 'AGD', 'Żywność', 'Elektronika', 'Żywność', 'Elektronika', 'Meble'],
    'Metoda płatności': ['Karta', 'Gotówka', 'Gotówka', 'Karta', 'Gotówka', 'Gotówka', 'Karta', 'Gotówka', 'Karta', 'Gotówka']
})

# 2. Dyskretyzacja danych ciągłych
# Dyskretyzacja atrybutu Cena
data['Cena_przedział'] = pd.cut(data['Cena'], bins=[0, 89, 159, 229, 300], labels=['[20,89]', '[90,159]', '[160,229]', '[230,300]'])
# Dyskretyzacja atrybutu Liczba produktów
data['Liczba_produktow_przedział'] = pd.cut(data['Liczba produktów'], bins=[0, 2, 4, 6], labels=['[1,2]', '[3,4]', '[5,6]'])

# 3. Przygotowanie transakcji - każda transakcja jako lista zdyskretyzowanych cech
transactions = data[['Cena_przedział', 'Liczba_produktow_przedział', 'Kategoria', 'Metoda płatności']].apply(lambda row: row.tolist(), axis=1).tolist()

# Konwersja danych w transakcjach na typ string
transactions = [[str(item) for item in transaction] for transaction in transactions]

# 4. Przekształcenie danych na format macierzy binarnej
te = TransactionEncoder()
te_data = te.fit(transactions).transform(transactions)

# Określenie docelowej kolejności kolumn
columns_order = [
    '[20,89]', '[90,159]', '[160,229]', '[230,300]',
    '[1,2]', '[3,4]', '[5,6]',                   
    'Elektronika', 'Meble', 'Żywność', 'AGD',  
    'Karta', 'Gotówka'                       
]

# Tworzenie macierzy binarnej z zachowaniem określonej kolejności
basket = pd.DataFrame(te_data, columns=te.columns_)[columns_order]
basket.index += 1

basket = basket.astype(int)

# Wyświetlenie macierzy transakcji
print("Macierz binarna transakcji:")
print(basket)

basket = basket.astype(bool)

# 5. Znalezienie zbiorów częstych
frequent_itemsets = apriori(basket, min_support=0.4, use_colnames=True)

# 6. Generowanie reguł asocjacyjnych
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

# 7. Mapowanie etykiet na liczby
mapping = {label: idx + 1 for idx, label in enumerate(basket.columns)}

# Zmiana etykiet w zbiorach częstych
frequent_itemsets['itemsets'] = frequent_itemsets['itemsets'].apply(lambda x: frozenset([mapping[item] for item in x]))

print("\nZbiory częste:\n", frequent_itemsets)
print("\nReguły asocjacyjne:\n", rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

Macierz binarna transakcji:
    [20,89]  [90,159]  [160,229]  [230,300]  [1,2]  [3,4]  [5,6]  Elektronika  \
1         1         0          0          0      0      1      0            1   
2         0         1          0          0      1      0      0            0   
3         1         0          0          0      0      0      1            0   
4         1         0          0          0      1      0      0            1   
5         0         0          0          1      1      0      0            0   
6         1         0          0          0      0      1      0            0   
7         0         1          0          0      1      0      0            1   
8         1         0          0          0      0      0      1            0   
9         1         0          0          0      0      1      0            1   
10        0         0          1          0      1      0      0            0   

    Meble  Żywność  AGD  Karta  Gotówka  
1       0        0    0      1        